In [17]:
from langchain_community.document_loaders  import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 
import os 
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np
import uuid 
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any , Tuple

lod pdf documents

In [7]:
def processAllPDFs(pdf_directory):
    all_docs =[]
    pdf_path = Path(pdf_directory) # set the path 
    pdf_files = list(pdf_path.glob("**/*.pdf")) # fetches only pdfs

    print(f"Files: {len(pdf_files)}")

    for file in pdf_files:
        print(f"processing {file}...")
        try:
            loader = PyPDFLoader(str(file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = file.name, 
                doc.metadata['file_type'] ="pdf"

            all_docs.extend(documents)
            print(f"loaded: {len(documents)} pages")
            
        except Exception  as e:
            print(f"Error {e}")
    
    return all_docs

In [8]:
all_documents = processAllPDFs("pdfs")

Files: 2
processing pdfs/3545945.3569859.pdf...
loaded: 7 pages
processing pdfs/Linguodidactica_22_J_Ostanina-Olszewska_Modern_technology_in_language_learning_and_teaching.pdf...
loaded: 12 pages


In [9]:
all_documents

[Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'creator': 'LaTeX with acmart 2021/09/24 v1.80 Typesetting articles for the Association for Computing Machinery and hyperref 2022-06-13 v7.00r Hypertext links for LaTeX', 'creationdate': '2023-01-25T21:39:02+00:00', 'author': 'David J. Malan', 'keywords': 'analogies, demonstrations, demos, memorable moments, metaphors, pedagogy, props, sets, teachable moments', 'moddate': '2023-12-05T15:38:56-05:00', 'pdfversion': '1.5', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '-  Social and professional topics  ->  Computational thinking.CS1.Computer science education.', 'title': 'Computer Science with Theatricality', 'trapped': '/False', 'source': 'pdfs/3545945.3569859.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': ('3545945.3569859.pdf',), 'file_type': 'pdf'}, page_content='Computer Scie

In [13]:
# chunking 
def split_documents(documents, chunk_size=1000, chunk_overlap = 200): # legal data use larger chunk size
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap, 
        length_function=len,
        separators=['\n\n','\n', " ", ""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"docuemnts {len(documents)}: chunks: {len(split_doc)}")
    return split_doc

In [14]:
chunks = split_documents(all_documents)

docuemnts 19: chunks: 88


In [19]:
class EmbeddingManager:
    def __init__(self, model_name:str="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None 
        self._load_model()

    def _load_model(self): # encapusalition
        try:
            self.model = SentenceTransformer(self.model_name)       
        except Exception as e:
            print(f"Error: {e}")
            raise 

    def generate_embedding(self, texts:List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loading")
        
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

In [20]:
embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1380.79it/s]


In [21]:
embedding_manager